In [ ]:
from pathlib import Path
import os
import subprocess

def detect_repo_dir():
    colab_root = Path('/content')
    if colab_root.exists() and os.access(colab_root, os.W_OK):
        return colab_root / 'Helios', True
    try:
        top = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
        return Path(top), False
    except Exception:
        return Path.cwd(), False

REPO_DIR, IS_COLAB = detect_repo_dir()
os.environ['HELIOS_REPO_DIR'] = str(REPO_DIR)
os.environ['HELIOS_IS_COLAB'] = '1' if IS_COLAB else '0'
print(f'repo_dir={REPO_DIR}')
print(f'is_colab={IS_COLAB}')

In [ ]:
!nvidia-smi || true

from pathlib import Path
import os
import subprocess

repo_dir = Path(os.environ['HELIOS_REPO_DIR'])
repo_url = 'https://github.com/aryanputta/Helios.git'

if os.environ.get('HELIOS_IS_COLAB') == '1' and not (repo_dir / '.git').exists():
    subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
else:
    print(f'[cached] using existing repo: {repo_dir}')

In [ ]:
%cd $HELIOS_REPO_DIR
!git log --oneline -1
!git status --short
!ls

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${HELIOS_REPO_DIR:?missing HELIOS_REPO_DIR}"
cd "$REPO_DIR"

if [ "${HELIOS_IS_COLAB:-0}" = "1" ]; then
  bash scripts/setup_colab_cuda.sh | tee "$REPO_DIR/helios_setup.log"
else
  cmake -S . -B build \
    -DCMAKE_BUILD_TYPE=Release \
    -DHELIOS_BUILD_GOOGLE_BENCHMARK=OFF \
    -DHELIOS_BUILD_NVBENCH=OFF
  cmake --build build -- -j4
fi

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${HELIOS_REPO_DIR:?missing HELIOS_REPO_DIR}"
cd "$REPO_DIR"

bash scripts/live_proof_walkthrough.sh | tee "$REPO_DIR/live_proof_walkthrough.log"

In [ ]:
import json
import os
from pathlib import Path

repo_dir = Path(os.environ['HELIOS_REPO_DIR'])
compare = json.loads((repo_dir / 'results/json/bcsstk30_scalar_vs_threaded_walkthrough.json').read_text())
proof = json.loads((repo_dir / 'results/json/bcsstk30_spmv_walkthrough.json').read_text())
planner_lines = (repo_dir / 'results/json/planner_observations_walkthrough.jsonl').read_text().strip().splitlines()
planner_last = json.loads(planner_lines[-1]) if planner_lines else {}

print('raw_compare_json=', compare)
print('raw_proof_json=', proof)
print('raw_planner_log_last=', planner_last)

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${HELIOS_REPO_DIR:?missing HELIOS_REPO_DIR}"
cd "$REPO_DIR"

if [ "${HELIOS_IS_COLAB:-0}" = "1" ]; then
  bash scripts/run_colab_proof.sh | tee "$REPO_DIR/helios_run.log"
else
  bash scripts/repro_suite.sh | tee "$REPO_DIR/helios_run.log"
  python3 scripts/export_public_bundle.py --repo-root . --output-dir public_artifacts | tee -a "$REPO_DIR/helios_run.log"
fi

In [ ]:
!find $HELIOS_REPO_DIR/results/reports -maxdepth 1 -type f | sort

In [ ]:
from IPython.display import Image, display
import os

repo_dir = os.environ['HELIOS_REPO_DIR']
display(Image(f'{repo_dir}/results/reports/proof_dashboard.png'))
display(Image(f'{repo_dir}/results/reports/proof_latency_vs_scalar.png'))

In [ ]:
import os

repo_dir = os.environ['HELIOS_REPO_DIR']
print(open(f'{repo_dir}/results/reports/proof_report.md').read())

In [ ]:
!find $HELIOS_REPO_DIR/public_artifacts -maxdepth 4 -type f | sort

In [ ]:
!zip -r helios_public_artifacts.zip $HELIOS_REPO_DIR/public_artifacts > /dev/null
try:
    from google.colab import files
    files.download('helios_public_artifacts.zip')
except ImportError:
    print('Created helios_public_artifacts.zip in the current working directory.')

In [ ]:
# DEBUG — only run this if a proof cell failed
!tail -n 150 $HELIOS_REPO_DIR/live_proof_walkthrough.log || true
!tail -n 150 $HELIOS_REPO_DIR/helios_run.log || true